# Optimization and Regularization

This notebook covers:
- Overfitting vs underfitting diagnosis
- Dropout regularization technique
- Batch normalization for training stability
- Learning rate scheduling
- L2 regularization (weight decay)
- Data augmentation
- Early stopping

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

## Diagnosing Training Problems

Always diagnose before regularizing by examining training curves:

```
Pattern 1: Healthy training
  loss:     ↘↘↘ (plateau)
  val_loss: ↘↘↘ (plateau)
  Interpretation: Converged. Model is learning well.

Pattern 2: OVERFITTING (memorization)
  loss:     ↘↘↘↘↘↘↘
  val_loss: ↘↘↗↗↗↗↗
  Interpretation: Training loss ↓ but val loss ↑. Model memorizing training data.
  Solution: Add regularization (dropout, BatchNorm, L2), reduce model size, more data

Pattern 3: UNDERFITTING (insufficient capacity)
  loss:     ↘↘↘  (still decreasing)
  val_loss: ↘↘↘  (still decreasing)
  Interpretation: Both losses improving but haven't converged.
  Solution: Train longer, increase model size, more features

Pattern 4: Learning rate too high
  loss:     ↗↘↗↘↗  (oscillating)
  val_loss: ↗↘↗↘↗
  Interpretation: Wildly unstable gradients.
  Solution: Reduce learning rate
```

In [ ]:
# Load Fashion-MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
X_train = X_train[..., np.newaxis] / 255.0
X_test = X_test[..., np.newaxis] / 255.0

print(f"Data loaded: Train {X_train.shape}, Test {X_test.shape}")

## Regularization Technique 1: Dropout

**Dropout**: Randomly zeros out fraction of activations during training. At test time, all neurons are active.

```python
tf.keras.layers.Dropout(0.5)  # Drop 50% of neurons randomly each batch
```

**Why it works:**
- Forces network to learn redundant representations
- No single neuron can rely on any other
- Ensemble effect: like training multiple sub-networks
- Common dropout rates: 0.2-0.5

In [ ]:
# Model WITHOUT regularization (prone to overfitting)
model_no_reg = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_no_reg.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Model WITH dropout
model_with_dropout = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28, 1)),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.3),  # Drop 30%
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(10, activation='softmax')
])

model_with_dropout.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

print("Model WITHOUT dropdown parameters:", model_no_reg.count_params())
print("Model WITH dropout parameters:", model_with_dropout.count_params())
print("(Dropout adds no parameters, just gates)")

In [ ]:
# Train both models
print("Training model WITHOUT regularization...")
hist_no_reg = model_no_reg.fit(X_train, y_train, epochs=15, batch_size=128,
                                validation_split=0.2, verbose=0)

print("Training model WITH dropout...")
hist_dropout = model_with_dropout.fit(X_train, y_train, epochs=15, batch_size=128,
                                       validation_split=0.2, verbose=0)

# Evaluate
test_no_reg = model_no_reg.evaluate(X_test, y_test, verbose=0)
test_dropout = model_with_dropout.evaluate(X_test, y_test, verbose=0)

print(f"\nModel WITHOUT regularization - Test Accuracy: {test_no_reg[1]:.4f}")
print(f"Model WITH dropout - Test Accuracy: {test_dropout[1]:.4f}")

In [ ]:
# Compare training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# No regularization
axes[0, 0].plot(hist_no_reg.history['loss'], label='Train', marker='o', markersize=3)
axes[0, 0].plot(hist_no_reg.history['val_loss'], label='Validation', marker='s', markersize=3)
axes[0, 0].set_title('WITHOUT Regularization - Loss')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(hist_no_reg.history['accuracy'], label='Train', marker='o', markersize=3)
axes[0, 1].plot(hist_no_reg.history['val_accuracy'], label='Validation', marker='s', markersize=3)
axes[0, 1].set_title('WITHOUT Regularization - Accuracy')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# With dropout
axes[1, 0].plot(hist_dropout.history['loss'], label='Train', marker='o', markersize=3)
axes[1, 0].plot(hist_dropout.history['val_loss'], label='Validation', marker='s', markersize=3)
axes[1, 0].set_title('WITH Dropout - Loss')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(hist_dropout.history['accuracy'], label='Train', marker='o', markersize=3)
axes[1, 1].plot(hist_dropout.history['val_accuracy'], label='Validation', marker='s', markersize=3)
axes[1, 1].set_title('WITH Dropout - Accuracy')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- WITHOUT regularization: Gap between train and val increases (overfitting)")
print("- WITH dropout: Train and validation curves stay closer (better generalization)")